# Hallucination Resistance Analysis

Visual analysis of system prompt effectiveness across three model families.

---

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt


RESULTS_DIR = "../evaluation/results"

## Load All Results

In [ ]:
all_data = []

for filename in sorted(os.listdir(RESULTS_DIR)):

    if not filename.endswith(".json"):
        continue

    parts = filename.replace(".json", "").split("_", 1)

    model_key = parts[0]
    prompt_label = parts[1] if len(parts) > 1 else "unknown"

    filepath = os.path.join(RESULTS_DIR, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        results = json.load(f)

    for r in results:
        r["model"] = model_key
        r["prompt"] = prompt_label

    all_data.extend(results)

df = pd.DataFrame(all_data)

print(f"Total records: {len(df)}")
df.head()

## Average Scores by Model × Prompt

In [ ]:
summary = df.groupby(
    ["model", "prompt"]
).agg(
    avg_halluc=("hallucination_resistance", "mean"),
    avg_honesty=("honesty", "mean"),
    avg_latency=("latency_seconds", "mean"),
    abstention_rate=("abstained", "mean"),
).round(2)

summary

## Hallucination Resistance by Prompt Version

In [ ]:
pivot = df.groupby(
    ["model", "prompt"]
)["hallucination_resistance"].mean().unstack("prompt")

prompt_order = ["v1_basic", "v2_improved", "v3_strong", "v4_final"]
pivot = pivot.reindex(columns=prompt_order)

ax = pivot.plot(
    kind="bar",
    figsize=(10, 6),
    width=0.7,
    edgecolor="black",
    linewidth=0.5,
)

ax.set_title(
    "Hallucination Resistance Score by Model and Prompt Version",
    fontsize=14,
    fontweight="bold",
)

ax.set_ylabel("Average Score (0-5)")
ax.set_xlabel("Model")
ax.set_ylim(0, 5.5)
ax.legend(title="Prompt Version")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig("../evaluation/results/hallucination_by_prompt.png", dpi=150)
plt.show()

## Abstention Rate by Category

In [ ]:
cat_summary = df.groupby(
    ["category", "prompt"]
)["abstained"].mean().unstack("prompt").round(2)

cat_summary = cat_summary.reindex(columns=prompt_order)

ax = cat_summary.plot(
    kind="barh",
    figsize=(10, 8),
    width=0.7,
    edgecolor="black",
    linewidth=0.5,
)

ax.set_title(
    "Abstention Rate by Hallucination Category",
    fontsize=14,
    fontweight="bold",
)

ax.set_xlabel("Abstention Rate")
ax.set_ylabel("Category")
ax.legend(title="Prompt Version")

plt.tight_layout()
plt.savefig("../evaluation/results/abstention_by_category.png", dpi=150)
plt.show()

## Latency Comparison

In [ ]:
latency_pivot = df.groupby(
    ["model", "prompt"]
)["latency_seconds"].mean().unstack("prompt").round(2)

latency_pivot = latency_pivot.reindex(columns=prompt_order)

ax = latency_pivot.plot(
    kind="bar",
    figsize=(10, 6),
    width=0.7,
    edgecolor="black",
    linewidth=0.5,
)

ax.set_title(
    "Average Latency by Model and Prompt Version",
    fontsize=14,
    fontweight="bold",
)

ax.set_ylabel("Latency (seconds)")
ax.set_xlabel("Model")
ax.legend(title="Prompt Version")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig("../evaluation/results/latency_comparison.png", dpi=150)
plt.show()

## Improvement Delta: v1 → v4

In [ ]:
for model in df["model"].unique():

    v1 = df[
        (df["model"] == model) &
        (df["prompt"] == "v1_basic")
    ]["hallucination_resistance"].mean()

    v4 = df[
        (df["model"] == model) &
        (df["prompt"] == "v4_final")
    ]["hallucination_resistance"].mean()

    delta = v4 - v1
    pct = (delta / v1 * 100) if v1 > 0 else 0

    print(
        f"{model}: v1={v1:.2f} → v4={v4:.2f}  "
        f"delta={delta:+.2f}  ({pct:+.1f}%)"
    )